# Sampling, Estimation, Confidence Intervals and the Bootstrap

Notebook for **Chapter 3** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch03.html

Tip: change a number, predict the output, then run the cell again.

## 3.1 Sampling Methods and Sampling Bias

In [ ]:
import numpy as np
rng = np.random.default_rng(20)
N = 100_000
seg = rng.random(N) < 0.7                       # True = mobile
spend = np.where(seg, rng.normal(20, 8, N), rng.normal(60, 15, N))
print("true mean:", round(spend.mean(), 3), " population sd:", round(spend.std(), 2))
mob, desk = spend[seg], spend[~seg]
srs, strat, biased = [], [], []
p_resp = np.where(seg, 1.0, 4.0); p_resp = p_resp / p_resp.sum()
for _ in range(2000):
    srs.append(rng.choice(spend, 200, replace=False).mean())
    strat.append(0.7 * rng.choice(mob, 140, replace=False).mean() + 0.3 * rng.choice(desk, 60, replace=False).mean())
    biased.append(spend[rng.choice(N, 200, replace=False, p=p_resp)].mean())
for name, a in [("simple random", srs), ("stratified", strat), ("biased (desktop x4)", biased)]:
    print(f"{name:22} mean of estimates={np.mean(a):7.3f}  sd of estimates={np.std(a):6.3f}")

true mean: 31.997  population sd: 21.11
simple random          mean of estimates= 31.990  sd of estimates= 1.473
stratified             mean of estimates= 31.979  sd of estimates= 0.742
biased (desktop x4)    mean of estimates= 45.212  sd of estimates= 1.622


In [ ]:
rng = np.random.default_rng(21)
depot_means = rng.normal(50, 10, 100)
drivers = depot_means[:, None] + rng.normal(0, 5, (100, 50))
flat = drivers.ravel()
a, b = [], []
for _ in range(2000):
    a.append(rng.choice(flat, 500, replace=False).mean())
    b.append(drivers[rng.choice(100, 10, replace=False)].mean())
print("true mean:", round(flat.mean(), 2))
print("random 500 drivers : sd of estimate =", round(np.std(a), 3))
print("10 whole clusters  : sd of estimate =", round(np.std(b), 3))

true mean: 50.51
random 500 drivers : sd of estimate = 0.424
10 whole clusters  : sd of estimate = 2.665


## 3.2 Standard Error and the Central Limit Theorem

In [ ]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(22)
mu_ln, sg_ln = 5.3, 0.5
pop_mean = np.exp(mu_ln + sg_ln**2 / 2)
pop_sd = pop_mean * np.sqrt(np.exp(sg_ln**2) - 1)
print("population mean %.1f  sd %.1f" % (pop_mean, pop_sd))
one = rng.lognormal(mu_ln, sg_ln, 200_000)
print("single-parcel skewness: %.2f" % stats.skew(one))
for n in (36, 400):
    means = rng.lognormal(mu_ln, sg_ln, (20_000, n)).mean(axis=1)
    print(f"n={n:<4} theory SE={pop_sd/np.sqrt(n):6.2f}  simulated SD of means={means.std():6.2f}  mean of means={means.mean():6.1f}  skew={stats.skew(means):5.2f}")

population mean 227.0  sd 121.0
single-parcel skewness: 1.77
n=36   theory SE= 20.16  simulated SD of means= 20.24  mean of means= 227.2  skew= 0.28
n=400  theory SE=  6.05  simulated SD of means=  6.08  mean of means= 227.0  skew= 0.10


In [ ]:
for n in (50, 500, 5000):
    ph = rng.binomial(n, 0.02, 50_000) / n
    th = (1 - 2 * 0.02) / np.sqrt(n * 0.02 * 0.98)
    print(f"n={n:<5} n*p={n*0.02:6.1f}  sim skew={stats.skew(ph):5.2f}  theory skew={th:5.2f}  SE={np.sqrt(0.02*0.98/n):.4f}")

n=50    n*p=   1.0  sim skew= 0.97  theory skew= 0.97  SE=0.0198
n=500   n*p=  10.0  sim skew= 0.32  theory skew= 0.31  SE=0.0063
n=5000  n*p= 100.0  sim skew= 0.08  theory skew= 0.10  SE=0.0020


In [ ]:
# plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(10, 3.2), sharey=False)
ax[0].hist(one[:20000], bins=60, range=(0, 900), color="#1f77b4"); ax[0].set_title("Single parcels (skewed)")
for a_, n in zip(ax[1:], (5, 100)):
    m = rng.lognormal(mu_ln, sg_ln, (20_000, n)).mean(axis=1)
    a_.hist(m, bins=60, color="#2ca02c", density=True)
    xs = np.linspace(m.min(), m.max(), 200)
    a_.plot(xs, stats.norm(pop_mean, pop_sd / np.sqrt(n)).pdf(xs), "k--")
    a_.set_title(f"Means of n={n} (dashed: normal)")
for a_ in ax: a_.set_xlabel("delivery time (min)")
plt.tight_layout(); plt.savefig(f"{IMG}/ch03_clt.png", dpi=120); plt.close()

## 3.3 Point Estimation: Maximum Likelihood and Method of Moments

In [ ]:
from scipy import optimize
x = np.array([2, 5, 1, 4, 3.0])
ll = lambda lam: len(x) * np.log(lam) - lam * x.sum()
res = optimize.minimize_scalar(lambda l: -ll(l), bounds=(0.01, 5), method="bounded")
print("MLE closed form:", round(len(x) / x.sum(), 4), " numeric:", round(res.x, 4))
print("SE:", round((len(x) / x.sum()) / np.sqrt(len(x)), 4))
print("loglik at 0.2 :", round(ll(0.2), 3), " at MLE:", round(ll(len(x) / x.sum()), 3))

MLE closed form: 0.3333  numeric: 0.3333
SE: 0.1491
loglik at 0.2 : -11.047  at MLE: -10.493


In [ ]:
rng = np.random.default_rng(23)
mom_k, mle_k = [], []
for _ in range(1000):
    d = rng.gamma(3, 50, 300)
    mom_k.append(d.mean()**2 / d.var(ddof=1))
    mle_k.append(stats.gamma.fit(d, floc=0)[0])
print("true shape 3.0")
print("MoM : mean %.3f  sd %.3f" % (np.mean(mom_k), np.std(mom_k)))
print("MLE : mean %.3f  sd %.3f" % (np.mean(mle_k), np.std(mle_k)))

true shape 3.0
MoM : mean 3.035  sd 0.278
MLE : mean 3.033  sd 0.234


In [ ]:
s5 = np.random.default_rng(24).standard_normal((200_000, 5))
print("divide by n  :", round(s5.var(axis=1, ddof=0).mean(), 4))
print("divide by n-1:", round(s5.var(axis=1, ddof=1).mean(), 4))

divide by n  : 0.8006
divide by n-1: 1.0008


## 3.4 Confidence Intervals

In [ ]:
n, xbar, s = 25, 480, 100
se = s / np.sqrt(n)
lo, hi = stats.t.interval(0.95, n - 1, loc=xbar, scale=se)
print("t interval: (%.1f, %.1f)  t*=%.3f" % (lo, hi, stats.t.ppf(0.975, n - 1)))
print("z interval: (%.1f, %.1f)" % (xbar - 1.96 * se, xbar + 1.96 * se))

t interval: (438.7, 521.3)  t*=2.064
z interval: (440.8, 519.2)


In [ ]:
r = stats.binomtest(18, 400)
print("Wald  :", np.round(0.045 + np.array([-1, 1]) * 1.96 * np.sqrt(0.045 * 0.955 / 400), 4))
print("Wilson:", np.round([r.proportion_ci(method="wilson").low, r.proportion_ci(method="wilson").high], 4))
print("Exact :", np.round([r.proportion_ci(method="exact").low, r.proportion_ci(method="exact").high], 4))
z = stats.binomtest(0, 200)
print("0 of 200 exact upper:", round(z.proportion_ci(method="exact").high, 4), " rule of three:", 3 / 200)

Wald  : [0.0247 0.0653]
Wilson: [0.0287 0.07  ]
Exact : [0.0269 0.0702]
0 of 200 exact upper: 0.0183  rule of three: 0.015


In [ ]:
rng = np.random.default_rng(25)
def coverage(n, reps=10_000):
    d = rng.lognormal(mu_ln, sg_ln, (reps, n))
    m, s = d.mean(1), d.std(1, ddof=1)
    tcrit = stats.t.ppf(0.975, n - 1)
    lo, hi = m - tcrit * s / np.sqrt(n), m + tcrit * s / np.sqrt(n)
    return ((lo <= pop_mean) & (pop_mean <= hi)).mean()
print("coverage n=15 :", coverage(15))
print("coverage n=100:", coverage(100))

coverage n=15 : 0.9286
coverage n=100: 0.9458


In [ ]:
# plot
d = rng.normal(480, 100, (40, 25))
m, s = d.mean(1), d.std(1, ddof=1); mg = stats.t.ppf(0.975, 24) * s / 5
fig, ax = plt.subplots(figsize=(6, 4.2))
for i in range(40):
    miss = not (m[i] - mg[i] <= 480 <= m[i] + mg[i])
    ax.plot([m[i] - mg[i], m[i] + mg[i]], [i, i], color="#d62728" if miss else "#1f77b4", lw=2)
ax.axvline(480, color="k", ls="--"); ax.set_xlabel("mean delivery time (min)"); ax.set_ylabel("repeated experiment #")
ax.set_title("40 repeated 95% intervals (red = missed the truth)")
plt.tight_layout(); plt.savefig(f"{IMG}/ch03_ci_coverage.png", dpi=120); plt.close()

## 3.5 The Bootstrap

In [ ]:
rng = np.random.default_rng(26)
x = rng.lognormal(5.3, 0.5, 500)
B = 10_000
idx = rng.integers(0, len(x), (B, len(x)))
boot = x[idx]
stats_b = {"mean": boot.mean(1), "median": np.median(boot, axis=1), "p95": np.percentile(boot, 95, axis=1)}
point = {"mean": x.mean(), "median": np.median(x), "p95": np.percentile(x, 95)}
for k, v in stats_b.items():
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"{k:6} point={point[k]:7.1f}  boot SE={v.std():6.2f}  95% CI=({lo:6.1f}, {hi:6.1f})")
print("analytic SE of mean s/sqrt(n):", round(x.std(ddof=1) / np.sqrt(len(x)), 2))
print("true median 200.0, true p95", round(np.exp(5.3 + 1.645 * 0.5), 1))

mean   point=  233.9  boot SE=  5.60  95% CI=( 223.2,  245.2)
median point=  208.7  boot SE=  6.32  95% CI=( 196.4,  219.8)
p95    point=  489.1  boot SE= 23.71  95% CI=( 438.4,  521.4)
analytic SE of mean s/sqrt(n): 5.58
true median 200.0, true p95 456.0


In [ ]:
rng = np.random.default_rng(27)
def diff_ci(n):
    a = rng.lognormal(np.log(200), 0.5, n); b = rng.lognormal(np.log(190), 0.5, n)
    ia = rng.integers(0, n, (5000, n)); ib = rng.integers(0, n, (5000, n))
    d = np.median(b[ib], axis=1) - np.median(a[ia], axis=1)
    return np.median(b) - np.median(a), np.percentile(d, [2.5, 97.5])
for n in (300, 1200):
    est, (lo, hi) = diff_ci(n)
    print(f"n={n:<5} diff of medians={est:6.1f}  95% CI=({lo:6.1f}, {hi:6.1f})  width={hi-lo:5.1f}")

n=300   diff of medians= -14.0  95% CI=( -33.8,    9.0)  width= 42.8
n=1200  diff of medians= -12.9  95% CI=( -22.7,   -1.5)  width= 21.2


## Chapter recap

## Practice set

Problem 3.6.1 A sample of 64 values has SD 16. What is the SE of the mean?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch03.html